# 01. Data Preprocessing

이 노트북은 여러 피부 데이터셋을 하나의 공통 학습 형식으로 합치는 단계입니다.

비유하면:

- 각 데이터셋은 서로 다른 학교에서 가져온 성적표
- 전처리 코드는 그 성적표를 같은 양식으로 다시 적는 교무실
- 최종 CSV는 모델이 읽는 공통 시험지


In [ ]:
!pip install -q -r requirements_colab.txt


In [ ]:
from google.colab import drive

# Google Drive를 연결하면 학습 중간 저장 파일이 세션 종료 후에도 남습니다.
# 비유하면 Colab 임시 책상 말고 개인 사물함에 저장하는 것입니다.
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

import pandas as pd

from src.skin_coach.preprocessing import (
    DatasetSpec,
    build_multimodal_targets,
    build_temporal_targets,
    integrate_image_datasets,
    standardize_daily_logs,
    standardize_user_profiles,
    write_preprocessed_artifacts,
)

# 프로젝트 폴더와 데이터 폴더를 먼저 정합니다.
# 비유하면 "오늘 수업할 교실"과 "교과서 창고"를 정하는 단계입니다.
PROJECT_ROOT = Path("/content/2026_DNN")
DATA_ROOT = Path("/content/data")
OUTPUT_DIR = PROJECT_ROOT / "processed"
DRIVE_ROOT = Path("/content/drive/MyDrive/2026_DNN")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, DATA_ROOT, OUTPUT_DIR, DRIVE_ROOT


In [ ]:
# 여기서는 어떤 데이터셋을 사용할지 목록을 적습니다.
# 비유하면 "어느 학교 성적표를 가져올지" 체크리스트를 만드는 셀입니다.
dataset_specs = [
    DatasetSpec(name="acne04", metadata_csv=str(DATA_ROOT / "acne04_metadata.csv"), image_root=str(DATA_ROOT / "images")),
    DatasetSpec(name="acne04v2", metadata_csv=str(DATA_ROOT / "acne04v2_metadata.csv"), image_root=str(DATA_ROOT / "images")),
    DatasetSpec(name="scin", metadata_csv=str(DATA_ROOT / "scin_metadata.csv"), image_root=str(DATA_ROOT / "images")),
    DatasetSpec(name="ddi", metadata_csv=str(DATA_ROOT / "ddi_metadata.csv"), image_root=str(DATA_ROOT / "images")),
    DatasetSpec(name="fitzpatrick17k", metadata_csv=str(DATA_ROOT / "fitzpatrick_metadata.csv"), image_root=str(DATA_ROOT / "images")),
    DatasetSpec(name="ffhq_wrinkle", metadata_csv=str(DATA_ROOT / "wrinkle_metadata.csv"), image_root=str(DATA_ROOT / "images")),
    DatasetSpec(name="custom", metadata_csv=str(DATA_ROOT / "custom_image_manifest.csv"), image_root=str(DATA_ROOT / "images")),
]

# 실제로 존재하는 CSV만 남깁니다.
dataset_specs = [spec for spec in dataset_specs if Path(spec.metadata_csv).exists()]
dataset_specs


In [ ]:
# 1단계: 이미지 데이터셋 통합
# 여러 성적표를 한 줄 표로 합치는 과정입니다.
image_labels_df = integrate_image_datasets(dataset_specs)

print("image_labels_df shape:", image_labels_df.shape)
display(image_labels_df.head())


In [ ]:
# 2단계: 생활습관 로그 정리
# 매일 쓴 일기장을 날짜 순서대로 반듯하게 정리하는 단계입니다.
daily_logs_csv = DATA_ROOT / "daily_logs.csv"
user_profiles_csv = DATA_ROOT / "user_profiles.csv"

daily_logs_df = standardize_daily_logs(str(daily_logs_csv), image_labels_df=image_labels_df)
user_profiles_df = standardize_user_profiles(str(user_profiles_csv) if user_profiles_csv.exists() else "")

print("daily_logs_df shape:", daily_logs_df.shape)
display(daily_logs_df.head())

print("user_profiles_df shape:", user_profiles_df.shape)
display(user_profiles_df.head())


In [ ]:
# 3단계: 시계열 타깃 생성
# "앞으로 피부가 나빠질까?" 같은 문제를 풀기 위해
# 과거 기록에서 미래 정답표를 만드는 단계입니다.
temporal_targets_df = build_temporal_targets(
    daily_logs_df=daily_logs_df,
    seq_len=14,
    worsening_threshold_points=5.0,
)

print("temporal_targets_df shape:", temporal_targets_df.shape)
display(temporal_targets_df.head())


In [ ]:
# 4단계: 멀티모달 타깃 생성
# 사진 성적표와 생활습관 일기장을 한 책상 위에 같이 놓는 느낌입니다.
multimodal_targets_df = build_multimodal_targets(
    image_labels_df=image_labels_df,
    daily_logs_df=daily_logs_df,
    temporal_targets_df=temporal_targets_df,
    user_profiles_df=user_profiles_df,
)

print("multimodal_targets_df shape:", multimodal_targets_df.shape)
display(multimodal_targets_df.head())


In [ ]:
# 5단계: 최종 CSV 저장
# 이제 모델이 읽을 수 있는 교과서 4권을 저장합니다.
artifacts = write_preprocessed_artifacts(
    output_dir=str(OUTPUT_DIR),
    image_labels_df=image_labels_df,
    daily_logs_df=daily_logs_df,
    temporal_targets_df=temporal_targets_df,
    multimodal_targets_df=multimodal_targets_df,
)

artifacts


## 출력 파일 설명

- `image_labels.csv`: 이미지 모델이 읽는 성적표
- `daily_logs.csv`: 날짜별 생활습관 로그
- `temporal_targets.csv`: 시계열 모델의 정답표
- `multimodal_targets.csv`: 이미지 + 시계열 + 정적 정보가 합쳐진 최종 표
